### Step 1: Import Libraries & API Keys

In [18]:
import os
from openai import OpenAI
from dotenv import load_dotenv
from IPython.display import Markdown, display
import gradio as gr
import json

load_dotenv()

OPENAI_API_KEY = os.getenv("OPENAI_API_KEY")

if OPENAI_API_KEY is None:
    raise Exception("API key is missing !!")


### Step 2: Set up Pushover

In [2]:
#step 2a -> Set up account in browser
#step 2b -> Set up the app on your iPhone, log into the same account
#step 2c -> In the browser create an "Application / API Token"
#step 2d -> Copy your User key and API Token into the .env file,
# like this but wthout the hastag symbols and with your own keys:
# PUSHOVER_USER = XXXXXXXXX
# PUSHOVER_TOKEN = XYYYYYYY

# Save changes to the .env file





In [19]:
load_dotenv()

True

In [20]:
pushover_user = os.getenv("PUSHOVER_USER")
pushover_token = os.getenv("PUSHOVER_TOKEN")
pushover_url = "https://api.pushover.net/1/messages.json"

In [21]:
# Test pushover

import requests

def send_notification(message: str):
    payload = {"user": pushover_user, "token": pushover_token, "message": message}
    requests.post(pushover_url, data=payload)

In [22]:
send_notification("Hi to myself from this amazing AI engineering Training !!")

### Step 3: Describe Pushover as an LLM Tool

In [23]:
send_notification_function = {
    "name": "send_notification",
    "description": "Sends a push notification to the user's phone via Pushover. User this to alert the user about important events, completed tasks, or time-snsitive information.",
    "parameters": {
        "type": "object",
        "properties": {
            "message": {
                "type": "string",
                "description": "The notification message to send to the user's device"
            }
        },
        "required": ['message']
    }
}

### Step 4: Add Pushover to the list of tools for the LLM

In [24]:
tools = [
    {"type": "function", "function": send_notification_function},
    #{Tool 2}
    #{Tool 3} .....
    ]

### Steps 2b & 3b & 4b: Create new function, describe it, add it

In [38]:
import random

# Simulate rolling a single die
def dice_roll():
    result = random.randint(1,6)
    return result


# Describe function for LLM
dice_roll_function = {
    "name": "dice_roll",
    "description": "Simulates rolling a single six-sided die and returns the result. Use this when the user wants to roll a die for games, decisions, or random number generation.",
    "parameters":{
        "type": "object",
        "properties": {},
        "required": []
    }
}

# Add function to list of tools of LLM
tools.append({"type" : "function", "function" : dice_roll_function})

### Step 5: Calling the tool from an LLM

In [39]:
# def handle_tool_call(tool_calls):
#     tool_call_results = []
#     for tool_call in tool_calls:
#         # tool_call = tool_calls[0] # Assuming just one tool call
#         args = json.loads(tool_call.function.arguments)

#         # Actually send the notification ie. call the tool !!
#         send_notification(args['message'])
#         #print(f"Sent notification: {args['message']}")

#         tool_call_result = {
#             "role" : "tool",
#             "tool_call_id" : tool_call.id,
#             "content" : f"Sent notification: {args['message']}"
#         }
#         tool_call_results.append(tool_call_result)
#     return tool_call_results



# Future-proofing - making the function more flexible across all tools

def handle_tool_call(tool_calls):
    tool_call_results = []
    for tool_call in tool_calls:
        function_name = tool_call.function.name
        args = json.loads(tool_call.function.arguments)
        # print(f"Calling function {function_name}") # For future debugging

        # Route to the appropriate function based on function_name
        if function_name == "send_notification":
            send_notification(args['message'])
            content = f"Sent notification: {args['message']}"
        
        # elif function_name == "func":
            # content = func2(args['message'])

        elif function_name == "dice_roll":
            content = f"Rolled: {dice_roll()}"

        else:
            content = f"Unknown function: {function_name}"

        tool_call_result = {
            "role" : "tool",
            "tool_call_id" : tool_call.id,
            "content" : content
        }
        tool_call_results.append(tool_call_result)
    return tool_call_results

In [ ]:
client = OpenAI()
# messages = [
#     {"role" : "user", "content" : "Please send two notifications telling me - 1) what amazing progress I'm making on the AI Engineering training by Super data science and 2) Hey there!"}
# ]

messages = [
    {"role" : "user", "content" : "Please do two things: \
                            1) I'd like to roll 2 dice, and \
                            2) Send me a notification with the highest of the rolls"}
]

response = client.chat.completions.create(
    model = "gpt-4.1-mini",
    messages=messages,
    tools= tools,
    tool_choice="auto" # options: none, required, default = auto
)

# Check if model wants to call a tool
message = response.choices[0].message

while message.tool_calls:
    from pprint import pprint
    pprint(message.tool_calls)
    # Handle the tool call
    tool_result = handle_tool_call(message.tool_calls) # Pass whole list 

    # Add info about tool call to context for LLM - ie. messages
    messages.append(message)
    messages.extend(tool_result)

    # Invoke LLM one more time to get its updated response
    response = client.chat.completions.create(
        model = "gpt-4.1-mini",
        messages=messages,
        tools = tools 
    )
    message = response.choices[0].message

    # Maybe consider adding protection from infinite consecutive tool calling

print(message.content)


[ChatCompletionMessageFunctionToolCall(id='call_VUY7RuLg4vIodt7j9DOdaxlg', function=Function(arguments='{}', name='dice_roll'), type='function'),
 ChatCompletionMessageFunctionToolCall(id='call_fUop47X14TnkMoaR1NTyuwfL', function=Function(arguments='{}', name='dice_roll'), type='function')]
[ChatCompletionMessageFunctionToolCall(id='call_usxDBmFBgx6KYPMvtQ3WyYm5', function=Function(arguments='{"message":"The highest roll among the two dice is 1."}', name='send_notification'), type='function')]
I rolled two dice and both came up as 1. I have sent you a notification with the highest roll, which is 1.
